##Setup

In [1]:
!pip install gymnasium shimmy ale-py
!pip install autorom
!AutoROM -y

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/assault.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asterix.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/asteroids.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/atlantis2.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/backgammon.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/bank_heist.bin
Inst

In [ ]:
from collections import defaultdict, deque
import numpy as np
import cv2
import gymnasium as gym
import ale_py
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# ──────────────────────────────────────────────
# Go-Explore cell functions (pixel-based)
# ──────────────────────────────────────────────
def cellfn(frame):
    cell = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    cell = cv2.resize(cell, (11, 8), interpolation=cv2.INTER_AREA)
    cell = cell // 32
    return cell

def hashfn(cell):
    return hash(cell.tobytes())

# ──────────────────────────────────────────────
# Go-Explore archive
# ──────────────────────────────────────────────
class Weights:
    times_chosen           = 0.1
    times_chosen_since_new = 0.0
    times_seen             = 0.3

class Powers:
    times_chosen           = 0.5
    times_chosen_since_new = 0.5
    times_seen             = 0.5

class Cell:
    def __init__(self):
        self.times_chosen           = 0
        self.times_chosen_since_new = 0
        self.times_seen             = 0

    def __setattr__(self, key, value):
        object.__setattr__(self, key, value)
        if key != 'score' and hasattr(self, 'times_seen'):
            self.score = self.cellscore()

    def cntscore(self, a):
        w = getattr(Weights, a)
        p = getattr(Powers, a)
        v = getattr(self, a)
        return w / (v + 1e-3) ** p + 1e-5

    def cellscore(self):
        return (self.cntscore('times_chosen') +
                self.cntscore('times_chosen_since_new') +
                self.cntscore('times_seen') + 1)

    def visit(self):
        self.times_seen += 1
        return self.times_seen == 1

    def choose(self):
        self.times_chosen           += 1
        self.times_chosen_since_new += 1
        return self.ram, self.reward, self.trajectory


# ──────────────────────────────────────────────
# Shared Actor-Critic network
# ──────────────────────────────────────────────
class ACERNet(nn.Module):
    """
    Outputs both a policy π(a|s) and per-action Q-values Q(s,a).
    The state value is recovered as V(s) = Σ_a π(a|s) Q(s,a).
    """
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),             nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),             nn.ReLU(),
        )
        with torch.no_grad():
            self._feature_size = self.conv(
                torch.zeros(1, *input_shape)
            ).view(1, -1).size(1)

        self.policy_head = nn.Linear(self._feature_size, n_actions)
        self.q_head      = nn.Linear(self._feature_size, n_actions)

    def forward(self, x):
        x        = x.float() / 255.
        features = self.conv(x).view(x.size(0), -1)
        pi       = F.softmax(self.policy_head(features), dim=-1)
        q        = self.q_head(features)
        return pi, q


# ──────────────────────────────────────────────
# ACER Agent
# ──────────────────────────────────────────────
class ACERAgent:
    def __init__(self, env, lr=1e-4, gamma=0.99, c=10.0,
                 entropy_coef=0.05, grad_clip=0.5):
        self.env       = env
        self.n_actions = env.action_space.n
        obs            = env.observation_space.shape
        self.input_shape = (obs[2], obs[0], obs[1])   # (C, H, W)

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f'Using device: {self.device}')

        self.model     = ACERNet(self.input_shape, self.n_actions).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

        self.gamma        = gamma
        self.c            = c             # IS truncation constant
        self.entropy_coef = entropy_coef
        self.grad_clip    = grad_clip

    def preprocess(self, state):
        return torch.from_numpy(state).permute(2, 0, 1).unsqueeze(0).to(self.device)

    def select_action(self, state):
        with torch.no_grad():
            pi, _ = self.model(self.preprocess(state))
        pi_np  = pi[0].cpu().numpy()
        action = np.random.choice(self.n_actions, p=pi_np)
        return action, pi_np

    def update(self, trajectory):
        """
        trajectory: list of (state, action, reward, next_state, done, old_pi)

        FIX: accepts next_states explicitly so we can compute proper Bellman
        targets without the broken v.roll(-1) approach.
        """
        if len(trajectory) < 2:
            return 0.0

        states, actions, rewards, next_states, dones, old_pis = zip(*trajectory)

        states_t      = torch.stack(
            [torch.from_numpy(s).permute(2, 0, 1) for s in states]
        ).to(self.device)                                                    # (T, C, H, W)
        next_states_t = torch.stack(
            [torch.from_numpy(s).permute(2, 0, 1) for s in next_states]
        ).to(self.device)                                                    # (T, C, H, W)

        actions_t = torch.tensor(actions, dtype=torch.long).to(self.device) # (T,)
        rewards_t = torch.tensor(rewards, dtype=torch.float32).to(self.device)
        dones_t   = torch.tensor(dones,   dtype=torch.float32).to(self.device)
        old_pis_t = torch.tensor(np.array(old_pis), dtype=torch.float32).to(self.device)

        # ── Current policy and Q-values ──────────────────────────────────────
        pi, q = self.model(states_t)                                        # (T, A), (T, A)
        v      = (pi * q).sum(1)                                            # V(s) = Σ π Q

        # ── FIX: Proper Bellman target using actual next states ──────────────
        # v.roll(-1) was circularly shifting values — the last state's
        # "next value" became the first state's value, corrupting all targets.
        with torch.no_grad():
            _, q_next  = self.model(next_states_t)                          # (T, A)
            pi_next, _ = self.model(next_states_t)
            v_next     = (pi_next * q_next).sum(1)                          # V(s')

        # FIX: zero out v_next at terminal transitions so we don't bootstrap
        # beyond the end of an episode
        q_target = rewards_t + self.gamma * v_next * (1.0 - dones_t)       # (T,)

        # ── Importance sampling weights ───────────────────────────────────────
        pi_a     = pi.gather(1, actions_t.unsqueeze(1)).squeeze(1)          # π(a|s)
        old_pi_a = old_pis_t.gather(1, actions_t.unsqueeze(1)).squeeze(1)   # μ(a|s)
        rho      = (pi_a / (old_pi_a + 1e-8)).detach()
        rho_cap  = torch.clamp(rho, max=self.c)

        # ── Advantage using truncated IS ─────────────────────────────────────
        q_a  = q.gather(1, actions_t.unsqueeze(1)).squeeze(1)               # Q(s,a)
        adv  = (q_a - v).detach()

        # ── Policy loss ───────────────────────────────────────────────────────
        log_pi_a    = torch.log(pi_a + 1e-10)
        policy_loss = -(rho_cap * adv * log_pi_a).mean()

        # ── FIX: Entropy bonus — prevents policy collapse on sparse rewards ──
        entropy = -(pi * torch.log(pi + 1e-10)).sum(1).mean()

        # ── Value loss (Huber — robust to large TD errors early in training) ──
        # FIX: MSE replaced with Huber; target is computed from next_states,
        # not from the circular roll of the current-state value tensor
        value_loss = F.huber_loss(q_a, q_target.detach(), delta=1.0)

        loss = policy_loss + 0.5 * value_loss - self.entropy_coef * entropy

        self.optimizer.zero_grad()
        loss.backward()
        # FIX: gradient clipping — IS-weighted gradients can be large
        nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
        self.optimizer.step()

        return loss.item()


# ──────────────────────────────────────────────
# State restoration
# ──────────────────────────────────────────────
def restore_env(env, ram):
    """
    Restore ALE state and return the current screen.
    Never calls env.reset() which would overwrite the restored state.
    """
    env.unwrapped.ale.restoreState(ram)
    return env.unwrapped.ale.getScreenRGB()   # (210, 160, 3) uint8


# ──────────────────────────────────────────────
# Hyperparameters
# ──────────────────────────────────────────────
WARMUP_ITERS    = 50     # Pure random Go-Explore before ACER training begins
STEPS_PER_ITER  = 100    # Go-Explore steps per iteration
EXPLORE_EPSILON = 0.5    # Probability of random action during Go-Explore steps

# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────
env      = gym.make('ALE/MontezumaRevenge-v5', render_mode='rgb_array')
agent    = ACERAgent(env)

# FIX: archive is now actually used — cells are visited, saved, and restored
archive  = defaultdict(Cell)
highscore  = 0
frames     = 0
iterations = 0
restore_cell = None

state, _ = env.reset()
score      = 0
trajectory = []

while True:
    found_new_cell = False
    episode_data   = []

    for step in range(STEPS_PER_ITER):
        use_random = iterations < WARMUP_ITERS or random.random() < EXPLORE_EPSILON

        if use_random:
            action   = env.action_space.sample()
            # Store a uniform dummy old_pi so IS ratio ≈ 1 for random steps;
            # these steps are off-policy so we rely on clipped rho to down-weight them
            old_pi   = np.ones(agent.n_actions, dtype=np.float32) / agent.n_actions
        else:
            action, old_pi = agent.select_action(state)

        next_state, reward, terminal, truncated, info = env.step(action)

        # FIX: detect life loss before storing done so Bellman targets are
        # correctly zeroed at the actual terminal step
        life_lost = info['lives'] < 6
        done      = terminal or truncated or life_lost

        episode_data.append((state, action, reward, next_state, done, old_pi))

        score += reward
        trajectory.append(action)
        frames += 1

        if score > highscore:
            highscore = score

        if done:
            # FIX: reset score and trajectory on death so archive updates
            # after this point don't carry stale cumulative values
            score      = 0
            trajectory = []
            state, _   = env.reset()
            break
        else:
            # ── Archive update ────────────────────────────────────────────────
            # FIX: archive was declared but never populated in the original
            cell_repr   = cellfn(next_state)
            cellhash    = hashfn(cell_repr)
            cell        = archive[cellhash]
            first_visit = cell.visit()

            cell_reward = getattr(cell, 'reward',     -1e9)
            cell_traj   = getattr(cell, 'trajectory', [])
            better  = score > cell_reward
            shorter = score == cell_reward and len(trajectory) < len(cell_traj)

            if first_visit or better or shorter:
                cell.ram        = env.unwrapped.ale.cloneState()
                cell.reward     = score
                cell.trajectory = trajectory.copy()
                cell.times_chosen           = 0
                cell.times_chosen_since_new = 0
                found_new_cell = True

            state = next_state

    # ── ACER update ───────────────────────────────────────────────────────────
    if iterations >= WARMUP_ITERS and len(episode_data) >= 2:
        loss = agent.update(episode_data)
    else:
        loss = 0.0

    if found_new_cell and restore_cell is not None:
        restore_cell.times_chosen_since_new = 0

    iterations += 1

    # ── Select next cell to restore from ──────────────────────────────────────
    # FIX: archive is now actually restored from, completing the Go-Explore loop
    if len(archive) > 0:
        scores      = np.array([c.score for c in archive.values()])
        hashes      = list(archive.keys())
        probs       = scores / scores.sum()
        chosen_hash = np.random.choice(hashes, p=probs)
        restore_cell = archive[chosen_hash]
        ram, score, trajectory = restore_cell.choose()
        state = restore_env(env, ram)
    else:
        state, _ = env.reset()
        score      = 0
        trajectory = []

    print(
        f"Iter: {iterations:5d} | Cells: {len(archive):5d} | "
        f"Frames: {frames:8d} | MaxReward: {highscore:.1f} | "
        f"Loss: {loss:.4f}"
    )

Streaming output truncated to the last 5000 lines.
Iter:  1217 | Cells:   223 | Frames:    52453 | MaxReward: 0.0 | Loss: -0.1446
Iter:  1218 | Cells:   223 | Frames:    52460 | MaxReward: 0.0 | Loss: -0.1446
Iter:  1219 | Cells:   223 | Frames:    52560 | MaxReward: 0.0 | Loss: -0.1444
Iter:  1220 | Cells:   223 | Frames:    52563 | MaxReward: 0.0 | Loss: -0.1444
Iter:  1221 | Cells:   223 | Frames:    52622 | MaxReward: 0.0 | Loss: -0.1445
Iter:  1222 | Cells:   223 | Frames:    52722 | MaxReward: 0.0 | Loss: -0.1443
Iter:  1223 | Cells:   223 | Frames:    52728 | MaxReward: 0.0 | Loss: -0.1446
Iter:  1224 | Cells:   223 | Frames:    52734 | MaxReward: 0.0 | Loss: -0.1449
Iter:  1225 | Cells:   223 | Frames:    52766 | MaxReward: 0.0 | Loss: -0.1445
Iter:  1226 | Cells:   223 | Frames:    52866 | MaxReward: 0.0 | Loss: -0.1446
Iter:  1227 | Cells:   223 | Frames:    52893 | MaxReward: 0.0 | Loss: -0.1442
Iter:  1228 | Cells:   223 | Frames:    52897 | MaxReward: 0.0 | Loss: -0.1446
I